In [ ]:
%reload_ext autoreload
%autoreload 2

# Introduction
This example code requires that you have followed all the steps in the [README](../README.md)

If you are opening this file in a code editor that supports Jupyter notebooks you can run each cell one by one and see the results.

## Define variables
In the code below you will have to define the variables that will be used in the code blocks below.

In [ ]:
# --------------------------------------------------
# Get environment variables
# --------------------------------------------------
import os
from dotenv import load_dotenv

# Load .env variables into environment
load_dotenv(override=True, dotenv_path=".env")

# Function to ensure that environment variables are loaded
def get_env_variable(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise ValueError(f"Environment variable {name} is not set")
    return value

# Load environment variables
dsb_data_api_base_url = get_env_variable("DSB_DATA_API_BASE_URL")
maskinporten_client_private_key_file_path = get_env_variable("MASKINPORTEN_CLIENT_PRIVATE_KEY_PATH")
maskinporten_client_key_id = get_env_variable("MASKINPORTEN_CLIENT_KEY_ID")
maskinporten_client_id = get_env_variable("MASKINPORTEN_CLIENT_ID")
maskinporten_audience = get_env_variable("MASKINPORTEN_AUDIENCE")
maskinporten_scope = get_env_variable("MASKINPORTEN_SCOPE")
maskinporten_resource = get_env_variable("MASKINPORTEN_RESOURCE")

In [ ]:
# --------------------------------------------------
# Retrieve access token from Maskinporten
# --------------------------------------------------
import lib.maskinporten as maskinporten

# Read private key file
private_key_bytes = open(maskinporten_client_private_key_file_path, "rb").read()

# Get access token
access_token = maskinporten.get_access_token(
    key_id=maskinporten_client_key_id,
    client_id=maskinporten_client_id,
    audience=maskinporten_audience,
    scope=maskinporten_scope,
    resource=maskinporten_resource,
    private_key=private_key_bytes
)

In [ ]:
# --------------------------------------------------
# Request dataset from DSB Data API
# --------------------------------------------------
import requests

response = requests.get(
    url=f"{dsb_data_api_base_url}/api/v1/datasets/finance_norway/fact_claim",
    headers={
        "Authorization": f"Bearer {access_token}"
    }
)


In [ ]:
# --------------------------------------------------
# Print response metadata
# --------------------------------------------------
response_meta = {
    " Request": "",
    "url": response.url,
    "status-code": response.status_code,
    "content-type": response.headers.get("Content-Type"),
    "content-length": response.headers.get("Content-Length"),
    "size (mb)": f"{int(response.headers.get('Content-Length', 0)) / (1024 * 1024):.2f} MB" if response.headers.get("Content-Length") else "N/A",
    " Rows": "",
    "rows": response.headers.get("X-Rows"),
    "is_all_data": response.headers.get("X-Rows") == response.headers.get("X-Total-Count"),
    "% of total": f"{(int(response.headers.get('X-Rows', 0)) / int(response.headers.get('X-Total-Count', 1)) * 100):.2f}%" if response.headers.get("X-Total-Count") else "N/A",
    " Pagination": "",
    "total rows in dataset": response.headers.get("X-Total-Count"),
    "page_size": response.headers.get("X-Page-Size"),
    "current_page": response.headers.get("X-Current-Page"),
    "total_pages": response.headers.get("X-Total-Pages"),
}
max_key_len = max(len(k) for k in response_meta)
max_value_len = max(len(str(v)) for v in response_meta.values() if v is not None)
for k, v in response_meta.items():
    if k.startswith(' '):
        print("=" * (max_key_len + max_value_len + 3))
        print(f"{k:<{max_key_len}}")
        print("=" * (max_key_len + max_value_len + 3))
    else:
        print(f"{k:<{max_key_len}} : {v}")

In [ ]:
# --------------------------------------------------
# Load dataset into pandas DataFrame
# --------------------------------------------------
import pandas as pd
from io import BytesIO
df = pd.read_parquet(
    path=BytesIO(response.content),
    dtype_backend="pyarrow"
)

In [ ]:
# Remove CompanyCode column
df = df.drop(columns=['CompanyCode'], errors='ignore')
df.info()

In [ ]:
# Create data directory if it doesn't exist
import os
os.makedirs("./data", exist_ok=True)

df.to_csv(f"./data/finance_norway_fact_claim_{pd.Timestamp.now().strftime('%Y%m%d')}.csv", index=False)